# Enrichment Analyses -> Knoblich Human Organoids (DirMult pseudotime, excitatory + inhibitory)

**Primary analysis**: Over-Representation Analysis (ORA) using `padj < 0.05` significant genes as the foreground  
**Supplementary**: GSEA on `-log10(p_value_chisq)` ranking, `scoreType = "pos"` (all scores ≥ 0)

**Why ORA rather than GSEA as the primary analysis:**

- GSEA requires a meaningful gene-level effect size to _order_ the ranking. In APA data the signed metrics  
  (Δψ, ρ_W) do not map cleanly onto gene expression fold-change — a 3'UTR shortening in one gene may  
  increase protein output while in another it represses it.
- We therefore avoid ranking genes by directionality. Using `-log10(p)` alone (`scoreType="pos"`) is valid  
  for GSEA but only recovers the most-significant end of the distribution; ORA with a fixed FDR threshold  
  is the statistically honest primary approach for this data type.
- GSEA is included as a sensitivity / cross-validation check.

**Custom ORA ridgeplots:**  
 For each enriched term we plot the `-log10(padj_chisq)` distribution of _all universe genes belonging  
 to that term_, using `ggridges`. This shows the density of signal within each term without requiring  
 GSEA-style ordered statistics, and is the publication-primary visualization.

**Databases**: GO BP, KEGG (hsa), Disease Ontology, DisGeNET, REACTOME, MSigDB Hallmark, HPO  
**Two lineages analysed separately**: excitatory neurons, inhibitory neurons  
**Conda env**: `R_downstream`


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
    REPO_ROOT <- normalizePath(getwd())
    while (!file.exists(file.path(REPO_ROOT, "environment", "downstr.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
        REPO_ROOT <- dirname(REPO_ROOT)
    }
    Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR <- file.path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE)
setwd(WORK_DIR)

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────────
library(tidyverse)
library(stringr)
# Enrichment
library(clusterProfiler)
library(org.Hs.eg.db)
library(DOSE)
library(enrichplot)
library(ReactomePA)
library(msigdbr)
# Plotting
library(ggplot2)
library(ggridges)
library(viridis)
library(UpSetR)


## Paths and output directories


In [ ]:
base_dir <- paste0(DATA_ROOT, "/3utr/organoids/knoblich/utrcalling")
cr_dir <- file.path(base_dir, "composite_ranking")
output_dir <- file.path(base_dir, "ORA")
fig_dir <- file.path(output_dir, "figures")

for (d in c(output_dir, fig_dir)) dir.create(d, showWarnings = FALSE, recursive = TRUE)

metric <- "neglog10pval_chisq"

input_files <- list(
  excitatory = file.path(cr_dir, "composite_ranking_optimized2_excitatory.csv"),
  inhibitory = file.path(cr_dir, "composite_ranking_optimized2_inhibitory.csv")
)
cat("Output directory:", output_dir, "\n")


## Load and pre-process data

### Gene-name cleaning

Some entries carry a leading `|` character (artifact from the UTR-peak annotation step).  
We strip leading `|` and whitespace from `gene_name`, then remove any row whose `gene_name`  
still contains `|` (readthrough / fusion loci — ambiguous, same rationale as comma-filtered entries).


In [ ]:
load_and_filter <- function(path, lineage_label) {
  dm <- readr::read_csv(path, show_col_types = FALSE)
  cat(sprintf("[%s] Loaded %d genes\n", lineage_label, nrow(dm)))

  # 1. Keep only converged genes
  dm <- dm %>% filter(converged == TRUE)
  cat(sprintf("[%s] After converged filter: %d\n", lineage_label, nrow(dm)))

  # 2. Remove multi-gene rows (comma-separated gene_id — ambiguous loci)
  dm <- dm %>% filter(!grepl(",", gene_id))
  cat(sprintf("[%s] After comma-in-gene_id filter: %d\n", lineage_label, nrow(dm)))

  # 3. Clean gene_name: strip leading | and whitespace
  n_leading_pipe <- sum(grepl("^\\|", dm$gene_name), na.rm = TRUE)
  dm$gene_name <- str_trim(sub("^\\|+", "", dm$gene_name))
  cat(sprintf("[%s] Stripped leading '|' from %d gene names\n", lineage_label, n_leading_pipe))

  # 4. Remove rows whose gene_name still contains | (readthrough / fusion loci)
  n_mid_pipe <- sum(grepl("\\|", dm$gene_name), na.rm = TRUE)
  dm <- dm %>% filter(!grepl("\\|", gene_name) | is.na(gene_name))
  cat(sprintf("[%s] Removed %d mid-pipe gene names\n", lineage_label, n_mid_pipe))

  # 5. Recompute BH-adjusted p-value on the filtered set (ensures consistency)
  dm$padj_chisq <- p.adjust(dm$p_value_chisq, method = "BH")

  # 6. Clamp p_value_chisq == 0 to the smallest non-zero value
  min_p <- min(dm$p_value_chisq[dm$p_value_chisq > 0], na.rm = TRUE)
  n_zero <- sum(dm$p_value_chisq == 0, na.rm = TRUE)
  dm$p_value_chisq[dm$p_value_chisq == 0] <- min_p
  if (n_zero > 0) cat(sprintf("[%s] Clamped %d p=0 values to %.2e\n", lineage_label, n_zero, min_p))

  # 7. Ranking score for GSEA
  dm$rank_score <- -log10(dm$p_value_chisq)
  dm <- arrange(dm, desc(rank_score))

  cat(sprintf(
    "[%s] Significant (padj_chisq < 0.05): %d / %d (%.1f%%)\n",
    lineage_label, sum(dm$padj_chisq < 0.05), nrow(dm),
    100 * mean(dm$padj_chisq < 0.05)
  ))
  cat(sprintf(
    "[%s] rank_score range: [%.2f, %.2f]\n\n",
    lineage_label, min(dm$rank_score), max(dm$rank_score)
  ))
  dm
}

dm_exc <- load_and_filter(input_files$excitatory, "excitatory")
dm_inh <- load_and_filter(input_files$inhibitory, "inhibitory")


## ID mapping (human)

The Knoblich data are human cells — no ortholog mapping is needed.  
We map Ensembl IDs → human ENTREZ IDs via `org.Hs.eg.db`.  
`gene_name` is already in HGNC symbol format.


In [ ]:
add_entrez <- function(dm, lineage_label) {
  dm$entrezid <- mapIds(
    org.Hs.eg.db,
    keys = dm$gene_id,
    column = "ENTREZID", keytype = "ENSEMBL", multiVals = "first"
  )
  n_mapped <- sum(!is.na(dm$entrezid))
  cat(sprintf(
    "[%s] ENTREZ mapped: %d / %d (%.1f%%)\n",
    lineage_label, n_mapped, nrow(dm), 100 * n_mapped / nrow(dm)
  ))
  dm
}

dm_exc <- add_entrez(dm_exc, "excitatory")
dm_inh <- add_entrez(dm_inh, "inhibitory")


## Build named score vectors for GSEA


In [ ]:
make_score_vectors <- function(dm, lineage_label) {
  # By HGNC symbol (GO BP, Hallmark, HPO)
  v_sym <- setNames(dm$rank_score, dm$gene_name)
  v_sym <- v_sym[!is.na(names(v_sym)) & names(v_sym) != ""]
  v_sym <- v_sym[!duplicated(names(v_sym))]

  # By ENTREZ (DO, DisGeNET, REACTOME, KEGG)
  v_entrez <- setNames(dm$rank_score, dm$entrezid)
  v_entrez <- v_entrez[!is.na(names(v_entrez))]
  v_entrez <- v_entrez[!duplicated(names(v_entrez))]

  cat(sprintf(
    "[%s] Score vectors — symbol: %d, ENTREZ: %d\n",
    lineage_label, length(v_sym), length(v_entrez)
  ))
  list(symbol = v_sym, entrez = v_entrez)
}

scores_exc <- make_score_vectors(dm_exc, "excitatory")
scores_inh <- make_score_vectors(dm_inh, "inhibitory")


## Build MSigDB gene sets (human)


In [ ]:
# Hallmark
hallmark_sets <- msigdbr(species = "Homo sapiens", collection = "H")
hallmark_t2g <- hallmark_sets %>%
  dplyr::select(gs_name, gene_symbol) %>%
  distinct()
hallmark_t2n <- hallmark_t2g %>%
  dplyr::select(gs_name) %>%
  distinct() %>%
  mutate(gs_description = gs_name %>%
    str_remove("^HALLMARK_") %>% str_replace_all("_", " ") %>% str_to_title())

# HPO
hpo_sets <- msigdbr(species = "Homo sapiens", collection = "C5", subcollection = "HPO")
hpo_t2g <- hpo_sets %>%
  dplyr::select(gs_name, gene_symbol) %>%
  distinct()
hpo_t2n <- hpo_t2g %>%
  dplyr::select(gs_name) %>%
  distinct() %>%
  mutate(gs_description = gs_name %>%
    str_remove("^HP_") %>% str_replace_all("_", " ") %>% str_to_title())

cat("Hallmark gene sets:", n_distinct(hallmark_t2g$gs_name), "\n")
cat("HPO gene sets:", n_distinct(hpo_t2g$gs_name), "\n")


## Helper: shared plot theme


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10, repr.plot.res = 200)
options(warn = -1)

custom_theme <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(size = 12, hjust = 0.5),
    panel.grid.major.x = element_blank(),
    panel.grid.minor.x = element_blank(),
    panel.grid.major.y = element_line(linewidth = 0.05, color = "grey60")
  )

# Save helper: saves as both PNG and PDF at 400 dpi
save_plot <- function(p, name, width = 8, height = 10) {
  for (ext in c("png", "pdf")) {
    ggsave(
      file.path(fig_dir, paste0(name, ".", ext)),
      plot = p, dpi = 400, width = width, height = height
    )
  }
  invisible(p)
}

# Custom ORA ridgeplot: for each enriched term, plot the -log10(padj_chisq)
# distribution of ALL universe genes belonging to that term.
# This is the PRIMARY publication visualization.
ora_ridgeplot <- function(ora_res, dm_all, score_col = "rank_score",
                          id_col = "gene_name", n_top = 20,
                          title = "", padj_cut = 0.05) {
  res_df <- as.data.frame(ora_res)
  sig <- res_df %>%
    filter(p.adjust < padj_cut) %>%
    arrange(p.adjust) %>%
    head(n_top)
  if (nrow(sig) == 0) {
    message("No significant terms (padj < ", padj_cut, ") — skipping ridgeplot")
    return(invisible(NULL))
  }

  # For each term, extract the genes listed in 'geneID' (separated by '/')
  # and look up their scores in dm_all
  score_map <- setNames(dm_all[[score_col]], dm_all[[id_col]])

  ridge_df <- sig %>%
    mutate(term_label = str_wrap(Description, 40)) %>%
    mutate(term_label = factor(term_label, levels = rev(str_wrap(Description, 40)))) %>%
    rowwise() %>%
    mutate(genes = list(unlist(strsplit(geneID, "/")))) %>%
    ungroup() %>%
    unnest(genes) %>%
    mutate(score = score_map[genes]) %>%
    filter(!is.na(score))

  vline_x <- -log10(0.05)

  ggplot(ridge_df, aes(x = score, y = term_label, fill = p.adjust)) +
    geom_density_ridges(
      scale = 1.2, alpha = 0.85,
      quantile_lines = TRUE, quantiles = 2,
      jittered_points = FALSE
    ) +
    geom_vline(
      xintercept = vline_x, linetype = "dashed",
      color = "black", linewidth = 0.4
    ) +
    scale_fill_viridis(
      name = "ORA\np.adjust", option = "magma",
      direction = -1, limits = c(0, padj_cut)
    ) +
    xlab("-log10(p_value_chisq) per gene") +
    ylab(NULL) +
    ggtitle(title) +
    custom_theme +
    theme(legend.position = "right")
}


---

# ORA — Primary Analysis

Foreground: genes with `padj_chisq < 0.05` (BH, recomputed on converged-only subset)  
Background: all converged tested genes (not the whole human genome — avoids inflating significance  
with genes that were never testable in this experiment)


## Run ORA — 7 databases × 2 lineages


In [ ]:
run_ora <- function(dm, scores, lineage_label, padj_cut = 0.05) {
  # Foreground / background gene lists
  sig_sym <- dm %>%
    filter(padj_chisq < padj_cut) %>%
    pull(gene_name) %>%
    na.omit() %>%
    unique()
  all_sym <- dm$gene_name[!is.na(dm$gene_name) & dm$gene_name != ""] %>% unique()

  sig_entrez <- dm %>%
    filter(padj_chisq < padj_cut) %>%
    pull(entrezid) %>%
    na.omit() %>%
    unique() %>%
    as.character()
  all_entrez <- dm$entrezid[!is.na(dm$entrezid)] %>%
    unique() %>%
    as.character()

  cat(sprintf(
    "[%s] ORA foreground: %d sym / %d ENTREZ | background: %d sym / %d ENTREZ\n",
    lineage_label, length(sig_sym), length(sig_entrez),
    length(all_sym), length(all_entrez)
  ))

  # ── GO Biological Process ──────────────────────────────────────────────────
  ora_go <- enrichGO(
    gene = sig_sym, universe = all_sym,
    OrgDb = org.Hs.eg.db, keyType = "SYMBOL", ont = "BP",
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA GO BP: %d sig terms\n",
    lineage_label, sum(ora_go@result$p.adjust < 0.05)
  ))

  # ── KEGG (human = hsa) ─────────────────────────────────────────────────────
  ora_kegg <- enrichKEGG(
    gene = sig_entrez, universe = all_entrez, organism = "hsa",
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  ora_kegg@result$Description <- gsub(
    " - Homo sapiens \\(human\\)$", "", ora_kegg@result$Description
  )
  cat(sprintf(
    "[%s] ORA KEGG: %d sig terms\n",
    lineage_label, sum(ora_kegg@result$p.adjust < 0.05)
  ))

  # ── Disease Ontology ───────────────────────────────────────────────────────
  ora_do <- enrichDO(
    gene = sig_entrez, universe = all_entrez,
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA DO: %d sig terms\n",
    lineage_label, sum(ora_do@result$p.adjust < 0.05)
  ))

  # ── DisGeNET ───────────────────────────────────────────────────────────────
  ora_dgn <- enrichDGN(
    gene = sig_entrez, universe = all_entrez,
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA DisGeNET: %d sig terms\n",
    lineage_label, sum(ora_dgn@result$p.adjust < 0.05)
  ))

  # ── REACTOME ───────────────────────────────────────────────────────────────
  ora_reactome <- enrichPathway(
    gene = sig_entrez, universe = all_entrez, organism = "human",
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA REACTOME: %d sig terms\n",
    lineage_label, sum(ora_reactome@result$p.adjust < 0.05)
  ))

  # ── MSigDB Hallmark ────────────────────────────────────────────────────────
  ora_hallmark <- enricher(
    gene = sig_sym, universe = all_sym,
    TERM2GENE = hallmark_t2g, TERM2NAME = hallmark_t2n,
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA Hallmark: %d sig terms\n",
    lineage_label, sum(ora_hallmark@result$p.adjust < 0.05)
  ))

  # ── HPO ────────────────────────────────────────────────────────────────────
  ora_hpo <- enricher(
    gene = sig_sym, universe = all_sym,
    TERM2GENE = hpo_t2g, TERM2NAME = hpo_t2n,
    pvalueCutoff = 0.05, pAdjustMethod = "BH",
    minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] ORA HPO: %d sig terms\n",
    lineage_label, sum(ora_hpo@result$p.adjust < 0.05)
  ))

  # Apply setReadable to ENTREZ-keyed results
  ora_do_r <- setReadable(ora_do, org.Hs.eg.db, "ENTREZID")
  ora_dgn_r <- setReadable(ora_dgn, org.Hs.eg.db, "ENTREZID")
  ora_kegg_r <- setReadable(ora_kegg, org.Hs.eg.db, "ENTREZID")
  ora_reactome_r <- setReadable(ora_reactome, org.Hs.eg.db, "ENTREZID")

  list(
    go = ora_go, kegg = ora_kegg, kegg_r = ora_kegg_r,
    do = ora_do, do_r = ora_do_r,
    dgn = ora_dgn, dgn_r = ora_dgn_r,
    reactome = ora_reactome, reactome_r = ora_reactome_r,
    hallmark = ora_hallmark, hpo = ora_hpo
  )
}

ora_exc <- run_ora(dm_exc, scores_exc, "excitatory")
ora_inh <- run_ora(dm_inh, scores_inh, "inhibitory")


## Sensitivity check: ORA at relaxed threshold (padj < 0.1)

Especially relevant for the inhibitory lineage (~5k cells, lower power).


In [ ]:
for (lin in c("excitatory", "inhibitory")) {
  dm_l <- if (lin == "excitatory") dm_exc else dm_inh
  n_05 <- sum(dm_l$padj_chisq < 0.05)
  n_10 <- sum(dm_l$padj_chisq < 0.10)
  cat(sprintf("[%s] sig genes: padj<0.05 = %d | padj<0.10 = %d\n", lin, n_05, n_10))
}


---

# GSEA — Supplementary Analysis

Ranking: `-log10(p_value_chisq)`, `scoreType = "pos"` (all non-negative).  
Serves as cross-validation of ORA results and captures terms enriched at the  
top of the significance distribution regardless of a fixed FDR threshold.


In [ ]:
run_gsea <- function(scores, lineage_label) {
  v_sym <- scores$symbol
  v_entrez <- scores$entrez

  # GO BP
  gsea_go <- gseGO(
    geneList = v_sym, ont = "BP", OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA GO BP: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_go@result),
    sum(gsea_go@result$p.adjust < 0.05)
  ))

  # Disease Ontology
  gsea_do <- gseDO(
    v_entrez,
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA DO: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_do@result),
    sum(gsea_do@result$p.adjust < 0.05)
  ))

  # DisGeNET
  gsea_dgn <- gseDGN(
    v_entrez,
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA DGN: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_dgn@result),
    sum(gsea_dgn@result$p.adjust < 0.05)
  ))

  # KEGG (human = hsa)
  gsea_kegg <- gseKEGG(
    geneList = v_entrez, organism = "hsa",
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  gsea_kegg@result$Description <- gsub(
    " - Homo sapiens \\(human\\)$", "", gsea_kegg@result$Description
  )
  cat(sprintf(
    "[%s] GSEA KEGG: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_kegg@result),
    sum(gsea_kegg@result$p.adjust < 0.05)
  ))

  # REACTOME
  gsea_reactome <- gsePathway(
    geneList = v_entrez, organism = "human",
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA REACTOME: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_reactome@result),
    sum(gsea_reactome@result$p.adjust < 0.05)
  ))

  # Hallmark
  gsea_hallmark <- GSEA(
    geneList = v_sym, TERM2GENE = hallmark_t2g, TERM2NAME = hallmark_t2n,
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA Hallmark: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_hallmark@result),
    sum(gsea_hallmark@result$p.adjust < 0.05)
  ))

  # HPO
  gsea_hpo <- GSEA(
    geneList = v_sym, TERM2GENE = hpo_t2g, TERM2NAME = hpo_t2n,
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )
  cat(sprintf(
    "[%s] GSEA HPO: %d terms (padj<0.05: %d)\n",
    lineage_label, nrow(gsea_hpo@result),
    sum(gsea_hpo@result$p.adjust < 0.05)
  ))

  # setReadable for ENTREZ-keyed results
  gsea_do_r <- setReadable(gsea_do, org.Hs.eg.db, "ENTREZID")
  gsea_dgn_r <- setReadable(gsea_dgn, org.Hs.eg.db, "ENTREZID")
  gsea_kegg_r <- setReadable(gsea_kegg, org.Hs.eg.db, "ENTREZID")
  gsea_reactome_r <- setReadable(gsea_reactome, org.Hs.eg.db, "ENTREZID")

  list(
    go = gsea_go,
    kegg = gsea_kegg, kegg_r = gsea_kegg_r,
    do = gsea_do, do_r = gsea_do_r,
    dgn = gsea_dgn, dgn_r = gsea_dgn_r,
    reactome = gsea_reactome, reactome_r = gsea_reactome_r,
    hallmark = gsea_hallmark,
    hpo = gsea_hpo
  )
}

gsea_exc <- run_gsea(scores_exc, "excitatory")
gsea_inh <- run_gsea(scores_inh, "inhibitory")


## GSEA sensitivity check: UMI ≥ 100 subset


In [ ]:
for (lin in c("excitatory", "inhibitory")) {
  dm_l <- if (lin == "excitatory") dm_exc else dm_inh
  gsea_l <- if (lin == "excitatory") gsea_exc else gsea_inh

  dm_hc <- dm_l %>% filter(UMIs >= 100)
  cat(sprintf("[%s] UMI>=100 subset: %d genes\n", lin, nrow(dm_hc)))

  v_hc <- setNames(dm_hc$rank_score, dm_hc$gene_name)
  v_hc <- v_hc[!is.na(names(v_hc)) & names(v_hc) != ""]
  v_hc <- v_hc[!duplicated(names(v_hc))]

  gsea_go_hc <- gseGO(
    geneList = v_hc, ont = "BP", OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
    pvalueCutoff = 0.5, pAdjustMethod = "BH",
    by = "fgsea", scoreType = "pos", minGSSize = 15, maxGSSize = 500
  )

  top_full <- head(gsea_l$go@result$Description[gsea_l$go@result$p.adjust < 0.05], 20)
  top_hc <- head(gsea_go_hc@result$Description[gsea_go_hc@result$p.adjust < 0.05], 20)
  cat(sprintf(
    "[%s] GO BP top-20 overlap (full vs UMI>=100): %d / %d\n",
    lin, length(intersect(top_full, top_hc)), max(length(top_full), 1)
  ))

  assign(paste0("gsea_go_hc_", lin), gsea_go_hc)
}


## Search for autism / neurodevelopmental terms


In [ ]:
neuro_terms <- "autis|neurodevelop|intellectual|ASD|speech|language|cogniti|synap|cortex|neural|FMRP|brain"

search_enrichment <- function(res_list, lineage_label, padj_cut = 0.5) {
  dbs <- c("go", "kegg", "do", "dgn", "reactome", "hallmark", "hpo")
  cat(sprintf("\n=== %s ===\n", lineage_label))
  for (db in dbs) {
    r <- res_list[[db]]
    hits <- r@result %>%
      filter(grepl(neuro_terms, Description, ignore.case = TRUE)) %>%
      arrange(p.adjust)
    if (nrow(hits) > 0) {
      cat(sprintf("  %s: %d matching terms\n", toupper(db), nrow(hits)))
      hits %>%
        dplyr::select(ID, Description, p.adjust, NES) %>%
        head(10) %>%
        print()
    }
  }
}

# Search GSEA results (pvalueCutoff=0.5 so all terms available)
search_enrichment(gsea_exc, "Excitatory")
search_enrichment(gsea_inh, "Inhibitory")

# Search ORA results (sig terms only)
cat("\n=== ORA significant terms matching neuro/autism ===\n")
for (lin in c("excitatory", "inhibitory")) {
  ora_l <- if (lin == "excitatory") ora_exc else ora_inh
  for (db in c("go", "kegg", "do", "dgn", "reactome", "hallmark", "hpo")) {
    hits <- ora_l[[db]]@result %>%
      filter(p.adjust < 0.05, grepl(neuro_terms, Description, ignore.case = TRUE))
    if (nrow(hits) > 0) {
      cat(sprintf("  [%s / %s] %d terms\n", lin, db, nrow(hits)))
    }
  }
}


---

# Save all results


In [ ]:
dbs <- c("go", "kegg", "do", "dgn", "reactome", "hallmark", "hpo")

for (lin in c("excitatory", "inhibitory")) {
  ora_l <- if (lin == "excitatory") ora_exc else ora_inh
  gsea_l <- if (lin == "excitatory") gsea_exc else gsea_inh
  for (db in dbs) {
    saveRDS(
      ora_l[[db]],
      file.path(output_dir, sprintf("ora_%s_%s_%s.rds", db, lin, metric))
    )
    saveRDS(
      gsea_l[[db]],
      file.path(output_dir, sprintf("gsea_%s_%s_%s.rds", db, lin, metric))
    )
  }
  # Score vectors
  scores_l <- if (lin == "excitatory") scores_exc else scores_inh
  saveRDS(scores_l$symbol, file.path(output_dir, sprintf("score_symbol_%s_%s.rds", lin, metric)))
  saveRDS(scores_l$entrez, file.path(output_dir, sprintf("score_entrez_%s_%s.rds", lin, metric)))
  # UMI>=100 GSEA GO sensitivity
  gsea_hc <- get(paste0("gsea_go_hc_", lin))
  saveRDS(gsea_hc, file.path(output_dir, sprintf("gsea_go_umi100_%s_%s.rds", lin, metric)))
}

cat("All results saved to", output_dir, "\n")


---

# Visualization

Run from here if results are already saved and the kernel has been restarted.


In [ ]:
# ── Load-back cell ─────────────────────────────────────────────────────────
# Re-run this cell to reload all saved objects without re-running the analyses.
library(tidyverse)
library(stringr)
library(clusterProfiler)
library(org.Hs.eg.db)
library(DOSE)
library(enrichplot)
library(ReactomePA)
library(msigdbr)
library(ggplot2)
library(ggridges)
library(viridis)
library(UpSetR)

base_dir <- paste0(DATA_ROOT, "/3utr/organoids/knoblich/utrcalling")
output_dir <- file.path(base_dir, "ORA")
fig_dir <- file.path(output_dir, "figures")
metric <- "neglog10pval_chisq"
dbs <- c("go", "kegg", "do", "dgn", "reactome", "hallmark", "hpo")

# Reload composite data (needed for ridgeplots)
cr_dir <- file.path(base_dir, "composite_ranking")
# (re-run load_and_filter + add_entrez cells if needed, or load from these CSVs)

for (lin in c("excitatory", "inhibitory")) {
  for (db in dbs) {
    assign(
      paste0("ora_", db, "_", lin),
      readRDS(file.path(output_dir, sprintf("ora_%s_%s_%s.rds", db, lin, metric)))
    )
    assign(
      paste0("gsea_", db, "_", lin),
      readRDS(file.path(output_dir, sprintf("gsea_%s_%s_%s.rds", db, lin, metric)))
    )
  }
  assign(
    paste0("score_symbol_", lin),
    readRDS(file.path(output_dir, sprintf("score_symbol_%s_%s.rds", lin, metric)))
  )
  assign(
    paste0("score_entrez_", lin),
    readRDS(file.path(output_dir, sprintf("score_entrez_%s_%s.rds", lin, metric)))
  )
}

cat("All results loaded\n")


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10, repr.plot.res = 200)
options(warn = -1)

custom_theme <- theme_minimal(base_size = 11) +
  theme(
    plot.title         = element_text(size = 12, hjust = 0.5),
    panel.grid.major.x = element_blank(),
    panel.grid.minor.x = element_blank(),
    panel.grid.major.y = element_line(linewidth = 0.05, color = "grey60")
  )

save_plot <- function(p, name, width = 8, height = 10) {
  for (ext in c("png", "pdf")) {
    ggsave(file.path(fig_dir, paste0(name, ".", ext)),
      plot = p, dpi = 400, width = width, height = height
    )
  }
  invisible(p)
}

ora_ridgeplot <- function(ora_res, dm_all, score_col = "rank_score",
                          id_col = "gene_name", n_top = 20,
                          title = "", padj_cut = 0.05) {
  res_df <- as.data.frame(ora_res)
  sig <- res_df %>%
    filter(p.adjust < padj_cut) %>%
    arrange(p.adjust) %>%
    head(n_top)
  if (nrow(sig) == 0) {
    message("No significant terms (padj < ", padj_cut, ") — skipping ridgeplot")
    return(invisible(NULL))
  }
  score_map <- setNames(dm_all[[score_col]], dm_all[[id_col]])
  ridge_df <- sig %>%
    mutate(term_label = str_wrap(Description, 40)) %>%
    mutate(term_label = factor(term_label, levels = rev(str_wrap(Description, 40)))) %>%
    rowwise() %>%
    mutate(genes = list(unlist(strsplit(geneID, "/")))) %>%
    ungroup() %>%
    unnest(genes) %>%
    mutate(score = score_map[genes]) %>%
    filter(!is.na(score))
  ggplot(ridge_df, aes(x = score, y = term_label, fill = p.adjust)) +
    geom_density_ridges(
      scale = 1.2, alpha = 0.85,
      quantile_lines = TRUE, quantiles = 2
    ) +
    geom_vline(
      xintercept = -log10(0.05), linetype = "dashed",
      color = "black", linewidth = 0.4
    ) +
    scale_fill_viridis(
      name = "ORA\np.adjust", option = "magma",
      direction = -1, limits = c(0, padj_cut)
    ) +
    xlab("-log10(p_value_chisq) per gene") +
    ylab(NULL) +
    ggtitle(title) +
    custom_theme +
    theme(legend.position = "right")
}


## ORA: Custom ridgeplots (primary publication figures)

Each ridge = distribution of `-log10(padj_chisq)` for all tested genes in that gene set.  
Dashed vertical line = `-log10(0.05)` significance threshold.  
Median line shown per ridge (quantile_lines = TRUE).


In [ ]:
# Reload dm data if not in environment (re-run if needed after kernel restart)
if (!exists("dm_exc")) {
  source_script <- function(path, lineage) {
    dm <- readr::read_csv(path, show_col_types = FALSE) %>%
      filter(converged == TRUE, !grepl(",", gene_id)) %>%
      mutate(gene_name = str_trim(sub("^\\|+", "", gene_name))) %>%
      filter(!grepl("\\|", gene_name) | is.na(gene_name))
    dm$padj_chisq <- p.adjust(dm$p_value_chisq, method = "BH")
    min_p <- min(dm$p_value_chisq[dm$p_value_chisq > 0], na.rm = TRUE)
    dm$p_value_chisq[dm$p_value_chisq == 0] <- min_p
    dm$rank_score <- -log10(dm$p_value_chisq)
    dm
  }
  cr_dir <- paste0(DATA_ROOT, "/3utr/organoids/knoblich/utrcalling/composite_ranking")
  dm_exc <- source_script(file.path(cr_dir, "composite_ranking_optimized2_excitatory.csv"), "excitatory")
  dm_inh <- source_script(file.path(cr_dir, "composite_ranking_optimized2_inhibitory.csv"), "inhibitory")
  cat("dm_exc and dm_inh reloaded\n")
}


In [ ]:
# ── GO BP ridgeplots ─────────────────────────────────────────────────────────
options(repr.plot.width = 10, repr.plot.height = 12, repr.plot.res = 200)

p_exc <- ora_ridgeplot(ora_go_excitatory, dm_exc,
  title = "GO BP — ORA (excitatory)\nDirMult pseudotime, Knoblich organoids"
)
p_inh <- ora_ridgeplot(ora_go_inhibitory, dm_inh,
  title = "GO BP — ORA (inhibitory)\nDirMult pseudotime, Knoblich organoids"
)

if (!is.null(p_exc)) {
  save_plot(p_exc, "ridgeplot_ora_go_excitatory", 10, 12)
  print(p_exc)
}
if (!is.null(p_inh)) {
  save_plot(p_inh, "ridgeplot_ora_go_inhibitory", 10, 12)
  print(p_inh)
}


In [ ]:
# ── KEGG ridgeplots ──────────────────────────────────────────────────────────
# KEGG geneID uses ENTREZ IDs; map back to symbols for the score_map
ora_ridgeplot_entrez <- function(ora_res, dm_all, n_top = 20, title = "", padj_cut = 0.05) {
  res_df <- as.data.frame(ora_res)
  sig <- res_df %>%
    filter(p.adjust < padj_cut) %>%
    arrange(p.adjust) %>%
    head(n_top)
  if (nrow(sig) == 0) {
    message("No sig terms — skipping")
    return(invisible(NULL))
  }
  score_map <- setNames(dm_all$rank_score, dm_all$entrezid)
  ridge_df <- sig %>%
    mutate(term_label = str_wrap(Description, 40)) %>%
    mutate(term_label = factor(term_label, levels = rev(str_wrap(Description, 40)))) %>%
    rowwise() %>%
    mutate(genes = list(unlist(strsplit(geneID, "/")))) %>%
    ungroup() %>%
    unnest(genes) %>%
    mutate(score = score_map[genes]) %>%
    filter(!is.na(score))
  ggplot(ridge_df, aes(x = score, y = term_label, fill = p.adjust)) +
    geom_density_ridges(scale = 1.2, alpha = 0.85, quantile_lines = TRUE, quantiles = 2) +
    geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4) +
    scale_fill_viridis(
      name = "ORA\np.adjust", option = "magma",
      direction = -1, limits = c(0, padj_cut)
    ) +
    xlab("-log10(p_value_chisq) per gene") +
    ylab(NULL) +
    ggtitle(title) +
    custom_theme +
    theme(legend.position = "right")
}

# Add entrezid back if needed
if (!"entrezid" %in% names(dm_exc)) {
  dm_exc$entrezid <- mapIds(org.Hs.eg.db, dm_exc$gene_id, "ENTREZID", "ENSEMBL", multiVals = "first")
  dm_inh$entrezid <- mapIds(org.Hs.eg.db, dm_inh$gene_id, "ENTREZID", "ENSEMBL", multiVals = "first")
}

p_exc <- ora_ridgeplot_entrez(ora_kegg_excitatory, dm_exc,
  title = "KEGG — ORA (excitatory)\nKnoblich organoids"
)
p_inh <- ora_ridgeplot_entrez(ora_kegg_inhibitory, dm_inh,
  title = "KEGG — ORA (inhibitory)\nKnoblich organoids"
)
if (!is.null(p_exc)) {
  save_plot(p_exc, "ridgeplot_ora_kegg_excitatory", 10, 10)
  print(p_exc)
}
if (!is.null(p_inh)) {
  save_plot(p_inh, "ridgeplot_ora_kegg_inhibitory", 10, 10)
  print(p_inh)
}


In [ ]:
# ── DO / DisGeNET / REACTOME ridgeplots (ENTREZ-keyed) ──────────────────────
for (db in c("do", "dgn", "reactome")) {
  for (lin in c("excitatory", "inhibitory")) {
    dm_l <- if (lin == "excitatory") dm_exc else dm_inh
    ora_l <- get(paste0("ora_", db, "_", lin))
    p_r <- ora_ridgeplot_entrez(ora_l, dm_l,
      n_top = 20,
      title = sprintf("%s — ORA (%s)\nKnoblich organoids", toupper(db), lin)
    )
    if (!is.null(p_r)) {
      save_plot(p_r, sprintf("ridgeplot_ora_%s_%s", db, lin), 10, 10)
      print(p_r)
    }
  }
}


In [ ]:
# ── Hallmark / HPO ridgeplots (symbol-keyed) ─────────────────────────────────
for (db in c("hallmark", "hpo")) {
  for (lin in c("excitatory", "inhibitory")) {
    dm_l <- if (lin == "excitatory") dm_exc else dm_inh
    ora_l <- get(paste0("ora_", db, "_", lin))
    p_r <- ora_ridgeplot(ora_l, dm_l,
      n_top = 20,
      title = sprintf("%s — ORA (%s)\nKnoblich organoids", toupper(db), lin)
    )
    if (!is.null(p_r)) {
      save_plot(p_r, sprintf("ridgeplot_ora_%s_%s", db, lin), 10, 10)
      print(p_r)
    }
  }
}


## ORA: Dotplots


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10, repr.plot.res = 200)

db_labels <- c(
  go = "GO BP", kegg = "KEGG", do = "Disease Ontology",
  dgn = "DisGeNET", reactome = "REACTOME",
  hallmark = "MSigDB Hallmark", hpo = "HPO"
)

for (db in dbs) {
  for (lin in c("excitatory", "inhibitory")) {
    ora_l <- get(paste0("ora_", db, "_", lin))
    n_sig <- sum(ora_l@result$p.adjust < 0.05)
    if (n_sig == 0) {
      cat(sprintf("[%s / %s] No sig ORA terms — skipping dotplot\n", db, lin))
      next
    }
    p_d <- dotplot(ora_l, showCategory = min(20, n_sig)) + custom_theme +
      ggtitle(sprintf("%s — ORA (%s)\nKnoblich organoids", db_labels[db], lin))
    save_plot(p_d, sprintf("dotplot_ora_%s_%s", db, lin))
    print(p_d)
  }
}


## ORA: Network plots (GO BP)


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10, repr.plot.res = 200)

for (lin in c("excitatory", "inhibitory")) {
  ora_go_l <- get(paste0("ora_go_", lin))
  score_l <- get(paste0("score_symbol_", lin))
  n_sig <- sum(ora_go_l@result$p.adjust < 0.05)
  if (n_sig < 2) {
    cat("Not enough terms for network plots\n")
    next
  }

  ora_go_pw <- pairwise_termsim(ora_go_l)

  p_emap <- emapplot(ora_go_pw, showCategory = min(30, n_sig)) +
    ggtitle(sprintf("GO BP ORA — Enrichment map (%s)", lin))
  save_plot(p_emap, sprintf("emapplot_ora_go_%s", lin), 10, 10)
  print(p_emap)

  p_cnet <- cnetplot(ora_go_pw, showCategory = 5, foldChange = score_l) +
    scale_color_viridis(name = "-log10(p)") +
    ggtitle(sprintf("GO BP ORA — Gene-concept network (%s)", lin))
  save_plot(p_cnet, sprintf("cnetplot_ora_go_%s", lin), 12, 10)
  print(p_cnet)

  if (n_sig >= 5) {
    p_tree <- treeplot(ora_go_pw, showCategory = min(30, n_sig)) +
      ggtitle(sprintf("GO BP ORA — Hierarchical clustering (%s)", lin))
    save_plot(p_tree, sprintf("treeplot_ora_go_%s", lin), 14, 10)
    print(p_tree)
  }
}


## GSEA: Ridgeplots (supplementary)


In [ ]:
options(repr.plot.width = 9, repr.plot.height = 10, repr.plot.res = 200)

for (db in dbs) {
  for (lin in c("excitatory", "inhibitory")) {
    gsea_l <- get(paste0("gsea_", db, "_", lin))
    n_sig <- sum(gsea_l@result$p.adjust < 0.05)
    if (n_sig == 0) {
      cat(sprintf("[%s / %s] No sig GSEA terms — skipping ridgeplot\n", db, lin))
      next
    }
    p_r <- ridgeplot(gsea_l,
      showCategory = min(20, n_sig),
      orderBy = "p.adjust", decreasing = TRUE
    ) +
      custom_theme +
      ggtitle(sprintf("%s — GSEA (%s)\nKnoblich organoids", db_labels[db], lin)) +
      xlab("-log10(p_value_chisq)") +
      geom_vline(
        xintercept = -log10(0.05), linetype = "dashed",
        color = "black", linewidth = 0.4
      )
    save_plot(p_r, sprintf("ridgeplot_gsea_%s_%s", db, lin), 9, 10)
    print(p_r)
  }
}


## GSEA: Dotplots + running-score plots (supplementary)


In [ ]:
options(repr.plot.width = 8, repr.plot.height = 10, repr.plot.res = 200)

for (db in dbs) {
  for (lin in c("excitatory", "inhibitory")) {
    gsea_l <- get(paste0("gsea_", db, "_", lin))
    n_sig <- sum(gsea_l@result$p.adjust < 0.05)
    if (n_sig == 0) next

    p_d <- dotplot(gsea_l, showCategory = min(20, n_sig)) + custom_theme +
      ggtitle(sprintf("%s — GSEA (%s)", db_labels[db], lin))
    save_plot(p_d, sprintf("dotplot_gsea_%s_%s", db, lin))
    print(p_d)

    n_top <- min(3, n_sig)
    p_gsea <- gseaplot2(gsea_l, geneSetID = 1:n_top, pvalue_table = TRUE)
    save_plot(p_gsea, sprintf("gseaplot_%s_top3_%s", db, lin), 10, 7)
    print(p_gsea)
  }
}


## GSEA: Network plots for GO BP (supplementary)


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10, repr.plot.res = 200)

for (lin in c("excitatory", "inhibitory")) {
  gsea_go_l <- get(paste0("gsea_go_", lin))
  score_l <- get(paste0("score_symbol_", lin))
  n_sig <- sum(gsea_go_l@result$p.adjust < 0.05)
  if (n_sig < 2) next

  gsea_go_pw <- pairwise_termsim(gsea_go_l)

  p_emap <- emapplot(gsea_go_pw, showCategory = min(30, n_sig)) +
    ggtitle(sprintf("GO BP GSEA — Enrichment map (%s)", lin))
  save_plot(p_emap, sprintf("emapplot_gsea_go_%s", lin), 10, 10)
  print(p_emap)

  p_cnet <- cnetplot(gsea_go_pw, showCategory = 5, foldChange = score_l) +
    scale_color_viridis(name = "-log10(p)") +
    ggtitle(sprintf("GO BP GSEA — Gene-concept network (%s)", lin))
  save_plot(p_cnet, sprintf("cnetplot_gsea_go_%s", lin), 12, 10)
  print(p_cnet)

  if (n_sig >= 5) {
    p_tree <- treeplot(gsea_go_pw, showCategory = min(30, n_sig)) +
      ggtitle(sprintf("GO BP GSEA — Hierarchical clustering (%s)", lin))
    save_plot(p_tree, sprintf("treeplot_gsea_go_%s", lin), 14, 10)
    print(p_tree)
  }
}


## Cross-lineage comparison: excitatory vs inhibitory

Within the Knoblich dataset: which enriched terms are shared across both lineages?


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8, repr.plot.res = 200)

compare_lineages_ora <- function(db, padj_cut = 0.05) {
  ora_e <- get(paste0("ora_", db, "_excitatory"))
  ora_i <- get(paste0("ora_", db, "_inhibitory"))

  sig_e <- ora_e@result %>%
    filter(p.adjust < padj_cut) %>%
    dplyr::select(ID, Description, p.adjust) %>%
    dplyr::rename(padj_exc = p.adjust)
  sig_i <- ora_i@result %>%
    filter(p.adjust < padj_cut) %>%
    dplyr::select(ID, Description, p.adjust) %>%
    dplyr::rename(padj_inh = p.adjust)

  overlap <- inner_join(sig_e, sig_i, by = c("ID", "Description"))

  cat(sprintf(
    "[%s] Exc: %d | Inh: %d | Shared: %d\n",
    db, nrow(sig_e), nrow(sig_i), nrow(overlap)
  ))

  # UpSet plot (UpSetR uses grid graphics — no title() after it)
  upset_list <- list(
    Excitatory = sig_e$ID,
    Inhibitory = sig_i$ID
  )
  upset_list <- upset_list[sapply(upset_list, length) > 0]
  if (length(upset_list) == 2) {
    upset(fromList(upset_list),
      order.by = "freq",
      text.scale = 1.3, point.size = 3, main.bar.color = "#3C5A8C",
      mainbar.y.label = sprintf("%s ORA — excitatory vs inhibitory", toupper(db)),
      sets.x.label = "Total sig. terms"
    )
  }

  # Scatter of shared terms
  if (nrow(overlap) >= 3) {
    p_sc <- ggplot(overlap, aes(
      x = -log10(padj_exc), y = -log10(padj_inh),
      label = str_wrap(Description, 30)
    )) +
      geom_point(alpha = 0.8, color = "#3C5A8C", size = 2) +
      ggrepel::geom_text_repel(size = 2.5, max.overlaps = 15) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "grey50") +
      xlab("-log10(p.adjust) excitatory") +
      ylab("-log10(p.adjust) inhibitory") +
      ggtitle(sprintf("%s ORA — shared terms\n(excitatory vs inhibitory)", toupper(db))) +
      custom_theme
    save_plot(p_sc, sprintf("scatter_ora_%s_exc_vs_inh", db), 8, 8)
    print(p_sc)
  }
  invisible(overlap)
}

# ggrepel for label overlaps
if (!requireNamespace("ggrepel", quietly = TRUE)) install.packages("ggrepel")
library(ggrepel)

for (db in dbs) compare_lineages_ora(db)
